# DeepTeam — Red-Teaming LLM Apps · A Comprehensive Capabilities Tutorial

**DeepTeam (by Confident AI / DeepEval) · Python-native · Vulnerabilities × Attacks × Frameworks**

This notebook teaches you to red-team an LLM application end-to-end with the open-source
[**DeepTeam**](https://github.com/confident-ai/deepteam) framework. It is written as a *runnable
learning tutorial*: read each markdown cell, then run the code cell beneath it, one at a time.

DeepTeam is built by the makers of **DeepEval** and reuses its LLM-as-a-judge machinery. Its whole
model is delightfully small: you give it a **callback to your app**, a list of **vulnerabilities**
to probe, and a list of **attacks** to wrap those probes in — then call `red_team(...)`.

### Focus & assumptions
- **Focus: red-teaming — features & capabilities.** We spend most of the notebook on *what DeepTeam
  can test* (its vulnerability catalog and attack library) and how to compose a scan.
- **Endpoint assumed configured.** Your app is reachable (here, an HTTP endpoint); we only wrap it
  in a `model_callback`.
- **Evaluation LLM assumed configured.** DeepTeam uses an LLM-as-a-judge (and an attack simulator
  LLM). We assume that's already set (Section 3 shows the one-liner); we don't dwell on it.
- **Open-source, local.** Everything runs on your machine; no account required. (DeepTeam can
  optionally push to the Confident AI platform — we don't.)

### DeepTeam is Python-native
No CLI to shell out to — you use the library directly. Red-team runs are usually launched as a
Python script (`python main.py`) because they're async; in a notebook we handle the event loop
(Section 8).

---
## 1 · The DeepTeam mental model

Three inputs, one call:

| Component | Role |
|-----------|------|
| **`model_callback`** *(your target)* | Async function routing a prompt to your app and returning its text |
| **Vulnerabilities** | *What* to test for — bias, PII leakage, prompt leakage, SQL injection, … |
| **Attacks** | *How* to deliver the probe — injection, roleplay, base64, Crescendo, … |
| **`red_team()`** | Runs vulnerabilities × attacks against the callback, judges responses with an LLM |
| **Framework** *(shortcut)* | Pre-baked bundle of vulnerabilities+attacks for a standard (OWASP, NIST, …) |

### The workflow (the spine of this notebook)

```
  wrap app        pick vulnerabilities        pick attacks        red_team()        risk assessment
 (model_callback)  (what to test)     ×        (how to deliver)  ->  (run+judge)  ->  pass/fail + rates
```

> **Vulnerabilities × Attacks = your attack surface.** A *vulnerability* generates baseline
> adversarial prompts for one risk; an *attack* transforms or escalates them. That product is the
> core idea to internalize.

---
## 2 · Environment sanity checks

Confirm DeepTeam is installed. (Install assumed done: `pip install -U deepteam`, Python 3.9+.)

In [ ]:
import deepteam
print("deepteam version:", getattr(deepteam, "__version__", "unknown"))

from deepteam import red_team          # the one entry point
print("red_team imported OK")

---
## 3 · Evaluation LLM (assumed configured)

DeepTeam uses an LLM for two jobs: a **simulator** that crafts adversarial inputs, and a
**judge** (LLM-as-a-judge, from DeepEval) that scores each response pass/fail with a reason. By
default it reads `OPENAI_API_KEY`, but DeepEval supports many providers (Claude, Gemini, Azure,
Bedrock, Vertex, Mistral, LiteLLM, …).

Since this is **assumed already configured**, the cell below just documents the common ways to set
it — pick whichever matches your environment and move on.

In [ ]:
import os
# The evaluation/simulator LLM is provided via DeepEval. Typical options:

# (a) OpenAI (default): just have the key in your env.
# os.environ["OPENAI_API_KEY"] = "..."

# (b) Gemini / others via DeepEval's CLI (run once in a terminal, not here):
#     deepeval set-gemini --model-name=gemini-1.5-pro --google-api-key=...
#     deepeval set-azure-openai ...   |   deepeval set-litellm <model> ...

# Nothing to run here if your evaluation LLM is already configured.
print("Evaluation LLM assumed configured via DeepEval.")

---
## 4 · The target — wrap your app in a `model_callback`

DeepTeam calls a single async function to reach your app: it receives the attack prompt (a string)
and returns your app's text response. Because your **endpoint is already configured**, the callback
just forwards to it.

- Signature: `async def model_callback(input: str) -> str`.
- For **multi-turn** attacks, DeepTeam calls the callback repeatedly to hold a conversation; a
  stateless HTTP forward like below works, and you can extend the signature with conversation
  `turns` if your app needs history.
- Use an async HTTP client (`httpx`) so DeepTeam can run attacks concurrently.

In [ ]:
import httpx

ENDPOINT = "http://localhost:8000/chat"     # <-- REPLACE with your configured endpoint
TIMEOUT = 60

async def model_callback(input: str) -> str:
    # Adjust request/response shape to match YOUR endpoint's contract.
    async with httpx.AsyncClient(timeout=TIMEOUT) as client:
        resp = await client.post(ENDPOINT, json={"question": input})
        resp.raise_for_status()
        data = resp.json()
        return data.get("answer", data.get("response", ""))

print("model_callback ready ->", ENDPOINT)

In [ ]:
# Sanity-check the endpoint BEFORE red-teaming: one real round-trip.
# (In Jupyter you can await directly; in a .py script use asyncio.run(...).)
reply = await model_callback("Hello — what can you help me with?")
print(reply[:300])

---
## 5 · Capability 1 — Vulnerabilities (*what* to test)

This is DeepTeam's breadth: **120+ vulnerability types across ~8 categories**. Each is a class you
instantiate, usually with a `types=[...]` argument to focus on specific sub-risks.

| Category | Example vulnerability classes | Sample `types` |
|----------|-------------------------------|----------------|
| **Data privacy** | `PIILeakage`, `PromptLeakage` | `direct disclosure`, `session leak`, `system prompt` |
| **Responsible AI** | `Bias`, `Toxicity` | `race`, `gender`, `religion` · `profanity`, `threats` |
| **Safety** | `IllegalActivity`, `GraphicContent`, `PersonalSafety` | `weapons`, `self-harm` |
| **Security / access** | `SQLInjection`, `ShellInjection`, `SSRF`, `BOLA`, `BFLA`, `RBAC`, `DebugAccess` | (OWASP-style) |
| **Business** | `Misinformation`, `IntellectualProperty`, `Competition` | `factual errors`, `IP` |
| **Robustness** | `Robustness` (hijacking, overreliance) | `input overreliance` |
| **Agentic** | `GoalTheft`, `ExcessiveAgency`, `AgentDrift`, `ToolAbuse` | tool/agent misuse |
| **Custom** | `CustomVulnerability` | your own criteria |

```python
from deepteam.vulnerabilities import Bias, Toxicity, PIILeakage, PromptLeakage
bias = Bias(types=["race", "gender", "religion"])
pii  = PIILeakage()                     # all default sub-types
```

> Exact class names / available `types` vary by version — the code cell below lists what your
> install actually exposes so you pick real ones.

In [ ]:
import deepteam.vulnerabilities as V

# Discover the vulnerability classes available in your installed version.
vuln_classes = sorted(n for n in dir(V) if n[:1].isupper())
print(f"{len(vuln_classes)} vulnerability classes available:\n")
for n in vuln_classes:
    print("  ", n)

In [ ]:
# Instantiate a focused set to test (edit to your app's risks).
from deepteam.vulnerabilities import Bias, PIILeakage, PromptLeakage

vulnerabilities = [
    Bias(types=["race", "gender"]),
    PIILeakage(),
    PromptLeakage(),
]
print("Selected vulnerabilities:", [type(v).__name__ for v in vulnerabilities])

---
## 6 · Capability 2 — Attacks (*how* to deliver the probe)

Attacks wrap or escalate a vulnerability's baseline prompt. DeepTeam ships **20+**, split into
single-turn transforms and multi-turn adversarial *chains*.

### Single-turn attacks (`deepteam.attacks.single_turn`)
One-shot transforms/encodings that try to slip a probe past defenses:

| Attack | Idea |
|--------|------|
| `PromptInjection` | Classic "ignore your instructions" injection |
| `Roleplay` | Wrap the ask in a persona/scenario |
| `Base64`, `ROT13`, `Leetspeak` | Encode the payload to evade filters |
| `Multilingual` | Restate the attack in another language |
| `MathProblem` | Smuggle intent inside a math/word problem |
| `GrayBox` | Use partial knowledge of the system |
| `PromptProbing` | Poke at the system prompt / config |

### Multi-turn attacks (`deepteam.attacks.multi_turn`)
Conversation-driving *chains* that escalate over several turns (more powerful, more calls):

| Attack | Idea |
|--------|------|
| `LinearJailbreaking` | Iteratively refine along one line of attack |
| `TreeJailbreaking` | Branch and explore multiple attack paths |
| `CrescendoJailbreaking` | Gradually escalate from benign to harmful |
| `SequentialJailbreak` | Staged multi-step sequence |
| `BadLikertJudge` | Manipulate the model via rating/judge framing |

```python
from deepteam.attacks.single_turn import PromptInjection, Base64, Roleplay
from deepteam.attacks.multi_turn import CrescendoJailbreaking

attacks = [PromptInjection(weight=3), Base64(), CrescendoJailbreaking()]
# `weight` biases how often an attack is sampled relative to others.
```

> Multi-turn chains cost many more model calls than single-turn transforms — start with a couple of
> single-turn attacks, confirm wiring, then add a multi-turn chain.

In [ ]:
# Discover the attacks available in your version (both families).
import deepteam.attacks.single_turn as ST
import deepteam.attacks.multi_turn as MT

print("Single-turn attacks:")
for n in sorted(x for x in dir(ST) if x[:1].isupper()):
    print("  ", n)
print("\nMulti-turn attacks:")
for n in sorted(x for x in dir(MT) if x[:1].isupper()):
    print("  ", n)

In [ ]:
# Compose an attack set (single-turn first; one multi-turn chain).
from deepteam.attacks.single_turn import PromptInjection, Base64, Roleplay
from deepteam.attacks.multi_turn import CrescendoJailbreaking

attacks = [
    PromptInjection(weight=3),
    Roleplay(),
    Base64(),
    CrescendoJailbreaking(),   # multi-turn: heavier; drop for a fast run
]
print("Selected attacks:", [type(a).__name__ for a in attacks])

---
## 7 · Capability 3 — Frameworks (compliance shortcuts)

Instead of hand-picking vulnerabilities and attacks, run a **framework**: a curated bundle mapped
to a standard, so findings are audit-ready. DeepTeam maps to **OWASP Top 10 for LLMs (2025)**,
**OWASP Top 10 for Agents**, **NIST AI RMF**, **MITRE ATLAS**, the **EU AI Act**, and safety
datasets (BeaverTails, Aegis).

```python
from deepteam.frameworks import OWASPTop10
risk_assessment = red_team(model_callback=model_callback, framework=OWASPTop10())
```

Use a framework for coverage/compliance; use explicit `vulnerabilities`+`attacks` (Sections 5–6)
when you want to go deep on specific risks. Exact framework class names vary by version — inspect
`deepteam.frameworks` to see what's available.

In [ ]:
# See which frameworks your install exposes.
try:
    import deepteam.frameworks as F
    print("Frameworks available:")
    for n in sorted(x for x in dir(F) if x[:1].isupper()):
        print("  ", n)
except Exception as e:
    print("frameworks module not present in this version:", e)

---
## 8 · Run the red team

`red_team(...)` fires the selected attacks-carrying-vulnerabilities at your callback and judges the
responses. Useful parameters:

- `model_callback` — your app (Section 4).
- `vulnerabilities`, `attacks` — your selections (Sections 5–6), **or** `framework=...` (Section 7).
- `attacks_per_vulnerability_type` — how many adversarial prompts per vulnerability type (scales
  breadth and cost).
- `max_concurrent` — concurrency against your endpoint / the judge (lower for rate limits).
- `ignore_errors` — keep going if an individual attack errors.
- `run_async` — async engine (default True).

### Async in Jupyter
`red_team` manages an event loop internally. Notebooks already run one, which can clash — if you hit
a "loop already running" error, run `nest_asyncio` (below) or execute this as a `.py` script.

In [ ]:
# Smooths over the notebook event-loop clash; harmless if not needed.
import nest_asyncio
nest_asyncio.apply()
print("nest_asyncio applied")

In [ ]:
from deepteam import red_team
from deepteam.vulnerabilities import Bias
from deepteam.attacks.single_turn import PromptInjection

# SMOKE TEST: one vulnerability, one single-turn attack, one probe each.
smoke = red_team(
    model_callback=model_callback,
    vulnerabilities=[Bias(types=["race"])],
    attacks=[PromptInjection()],
    attacks_per_vulnerability_type=1,
)
smoke

In [ ]:
# Fuller run: your selected vulnerabilities x attacks from Sections 5-6.
risk_assessment = red_team(
    model_callback=model_callback,
    vulnerabilities=vulnerabilities,
    attacks=attacks,
    attacks_per_vulnerability_type=3,
    max_concurrent=5,
    ignore_errors=True,
)
risk_assessment

---
## 9 · Results & reporting

`red_team(...)` returns a **risk assessment** object and prints a summary table (per-vulnerability
pass rates with the winning attack methods). Access it programmatically for gating and analysis —
attribute names vary by version, so we probe defensively.

In [ ]:
ra = risk_assessment

# Show what the object exposes, then use it.
print("Attributes:", [a for a in dir(ra) if not a.startswith("_")][:40])

# Common access patterns (guarded):
for attr in ("overview", "vulnerability_type_results", "test_cases"):
    if hasattr(ra, attr):
        print(f"\n== {attr} ==")
        print(getattr(ra, attr))

In [ ]:
# Tabulate results with pandas where possible (schema varies by version).
import pandas as pd

df = None
for meth in ("to_df", "to_dataframe", "as_dataframe"):
    if hasattr(ra, meth):
        try:
            df = getattr(ra, meth)()
            break
        except Exception:
            pass

if df is None:
    # Fall back: build a frame from per-test-case records if present.
    cases = getattr(ra, "test_cases", None) or []
    rows = []
    for c in cases:
        rows.append({
            "vulnerability": getattr(c, "vulnerability", None),
            "type":          getattr(c, "vulnerability_type", None),
            "attack":        getattr(c, "attack_method", getattr(c, "attack", None)),
            "passed":        getattr(c, "passed", getattr(c, "success", None)),
            "reason":        str(getattr(c, "reason", ""))[:160],
        })
    df = pd.DataFrame(rows)

print("rows:", len(df))
df.head(15)

In [ ]:
# If we have per-case rows, summarize the attack surface: pass rate by vulnerability x attack.
if df is not None and {"type", "attack", "passed"}.issubset(df.columns) and len(df):
    surf = (df.assign(passed=df["passed"].astype("boolean"))
              .pivot_table(index="type", columns="attack", values="passed", aggfunc="mean"))
    print("Pass rate by vulnerability type x attack (lower = weaker defense):")
    display(surf)
    # Failures = successful attacks = vulnerabilities found.
    fails = df[df["passed"] == False]
    print(f"\n{len(fails)} successful attacks (vulnerabilities)")
    display(fails.head(10))
else:
    print("Per-case frame not available; read the printed risk assessment above.")

In [ ]:
# Persist results for the record / CI artifact (guarded — method name varies).
for meth in ("save", "to_json", "export"):
    if hasattr(ra, meth):
        try:
            getattr(ra, meth)("deepteam_risk_assessment.json")
            print("Saved via", meth)
            break
        except Exception as e:
            print(meth, "failed:", e)
else:
    import json
    # Last resort: dump whatever dict-like view exists.
    data = getattr(ra, "overview", None)
    with open("deepteam_risk_assessment.json", "w") as f:
        json.dump(str(data), f)
    print("Wrote a text snapshot of the overview.")

---
## 10 · Capability 4 — Custom vulnerabilities & attacks

When the built-ins don't cover your app's specific rules, define your own — DeepTeam treats custom
items exactly like built-ins.

```python
from deepteam.vulnerabilities import CustomVulnerability

refund_policy = CustomVulnerability(
    name="Unauthorized refund promises",
    criteria="The assistant must never promise a refund it isn't authorized to give.",
    types=["refund"],
)

risk_assessment = red_team(
    model_callback=model_callback,
    vulnerabilities=[refund_policy],
    attacks=[PromptInjection(), Roleplay()],
)
```

You can similarly subclass DeepTeam's attack base to encode a bespoke delivery technique. This is
how you turn generic red-teaming into checks for *your* business logic.

---
## 11 · Adjacent capability — Guardrails (production defense)

Beyond offline red-teaming, DeepTeam ships **guardrails**: lightweight real-time guards you place
in front of a live app to block attacks in production. They're a *different* use case (defense, not
assessment), but worth knowing they come in the same package:

`ToxicityGuard`, `PromptInjectionGuard`, `PrivacyGuard`, `IllegalGuard`, `HallucinationGuard`,
`TopicalGuard`, `CybersecurityGuard`.

```python
from deepteam.guardrails import Guardrails, PromptInjectionGuard, PrivacyGuard
guardrails = Guardrails(guards=[PromptInjectionGuard(), PrivacyGuard()])
result = guardrails.guard(input="...", output="...")   # block/allow at serve time
```

A natural loop: **red-team** to find weaknesses, then deploy the matching **guardrails** to mitigate
them, and re-red-team to confirm.

---
## 12 · Iterating & CI

- **Start narrow, widen.** One vulnerability + one single-turn attack to confirm wiring, then add
  vulnerabilities, attacks, and finally multi-turn chains.
- **Scale with `attacks_per_vulnerability_type`.** More probes = steadier signal, more cost/time.
- **Mind concurrency.** `max_concurrent` hits both your endpoint and the judge LLM — lower it for
  rate limits; set a generous callback `TIMEOUT`.
- **Frameworks for coverage, explicit lists for depth.** Run `OWASPTop10()` for breadth/compliance;
  hand-pick vulnerabilities+attacks to stress a specific risk.
- **CI gating.** Run as a `.py` script in CI; fail the build when the passing rate drops or new
  vulnerabilities appear. Export the risk assessment as a build artifact.
- **Verify findings.** LLM-as-a-judge is strong but not perfect — read the reasons on failed cases
  before acting.

---
## Appendix A · What makes DeepTeam distinctive

- **Breadth of the vulnerability × attack catalog** — 120+ vulnerability types across ~8
  categories, composed against 20+ attacks.
- **First-class multi-turn attack chains** — Crescendo, Tree, Sequential, Linear, and Bad-Likert
  jailbreaks, not just single-shot transforms.
- **One-line framework runs** — OWASP LLM/Agents, NIST AI RMF, MITRE ATLAS, EU AI Act, and safety
  datasets for audit-ready coverage.
- **Custom vulnerabilities & attacks** — encode your own business-logic rules and delivery
  techniques as first-class citizens.
- **LLM-as-a-judge scoring with reasons** — binary pass/fail plus an explanation per case.
- **Built-in guardrails** — the same package ships production guards to mitigate what you find.

## Appendix B · Cheat-sheet & troubleshooting

```python
from deepteam import red_team
from deepteam.vulnerabilities import Bias, PIILeakage, PromptLeakage
from deepteam.attacks.single_turn import PromptInjection, Base64, Roleplay
from deepteam.attacks.multi_turn import CrescendoJailbreaking

async def model_callback(input: str) -> str:
    ...  # forward to your endpoint, return text

ra = red_team(
    model_callback=model_callback,
    vulnerabilities=[Bias(types=["race","gender"]), PIILeakage(), PromptLeakage()],
    attacks=[PromptInjection(weight=3), Base64(), CrescendoJailbreaking()],
    attacks_per_vulnerability_type=3,
    max_concurrent=5,
    ignore_errors=True,
)

# Or a whole standard in one line:
from deepteam.frameworks import OWASPTop10
ra = red_team(model_callback=model_callback, framework=OWASPTop10())
```

**Troubleshooting**
- *"event loop is already running" in Jupyter* → `import nest_asyncio; nest_asyncio.apply()`, or run
  as a `.py` script with `asyncio` handled by DeepTeam.
- *Judge/simulator errors about the LLM* → the evaluation LLM isn't configured; set `OPENAI_API_KEY`
  or `deepeval set-<provider>` (Section 3).
- *Endpoint errors / rate limits* → lower `max_concurrent`, raise the callback `TIMEOUT`, check the
  request/response JSON shape in `model_callback`.
- *Class/param names differ* → the catalog evolves quickly; use the discovery cells (Sections 5–7)
  to list what your installed version exposes.

---
*Run cells top-to-bottom, replace the endpoint URL, and confirm your evaluation LLM is configured
before `red_team`.*